# Baseline Results

Model:
- Linear Regression

Validation Strategy:
- Time-based split
- Train: 2013-2016
- Validation: 2017

Results:
- RMSLE = 2.1040

Observations:
- Pipeline works correctly.
- Performance leaves significant room for improvement.
- Nonlinear models are expected to outperform Linear Regression.

Cell 1

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(r"C:\Users\NickK\OneDrive\Υπολογιστής\github\kaggle_store_sales_forecasting")
PROCESSED_DIR = BASE_DIR / "data" / "processed"

train = pd.read_csv(PROCESSED_DIR / "train_features.csv")
test = pd.read_csv(PROCESSED_DIR / "test_features.csv")

print(train.shape)
print(test.shape)

(3000888, 21)
(28512, 20)


Cell 2 - Select features

In [2]:
target = "sales"

features = [
    "store_nbr",
    "family",
    "onpromotion",
    "year",
    "month",
    "day",
    "quarter",
    "week",
    "day_of_week",
    "is_weekend",
    "city",
    "state",
    "type",
    "cluster",
    "dcoilwtico",
    "is_payday",
    "is_national_holiday",
    "earthquake_period"
]

X = train[features]
y = train[target]
X_test = test[features]

print(X.shape)
print(y.shape)
print(X_test.shape)

(3000888, 18)
(3000888,)
(28512, 18)


Cell 3 - Encode categorical variables

In [3]:
X = pd.get_dummies(X, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)

print(X.shape)
print(X_test.shape)

(3000888, 86)
(28512, 86)


Cell 4

In [4]:
train["date"] = pd.to_datetime(train["date"])

split_date = "2017-01-01"

train_mask = train["date"] < split_date
valid_mask = train["date"] >= split_date

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_valid = X.loc[valid_mask]
y_valid = y.loc[valid_mask]

print(X_train.shape)
print(X_valid.shape)

(2596374, 86)
(404514, 86)


Cell 5 - First Baseline Model

In [5]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

print("Model trained.")

Model trained.


Cell 6 - Validation Predictions

In [6]:
valid_preds = model.predict(X_valid)

print(valid_preds[:10])

[ 139.47004971  134.19617017  136.33733892 2201.27398826  134.165759
  535.23001606  140.37238561 1092.31886363  719.4232132   298.06493895]


Cell 7 - RMSLE Evaluation

In [7]:
from sklearn.metrics import root_mean_squared_log_error

rmsle = root_mean_squared_log_error(
    y_valid,
    np.maximum(valid_preds, 0)
)

print("Validation RMSLE:", rmsle)

Validation RMSLE: 2.104005543565126


Cell 8 - First Kaggle Submission

In [8]:
test_preds = model.predict(X_test)

test_preds = np.maximum(test_preds, 0)

submission = pd.DataFrame({
    "id": test["id"],
    "sales": test_preds
})

submission.head()

,id,sales
0,3000888,88.193864
1,3000889,82.919984
2,3000890,119.944197
3,3000891,2498.828240
4,3000892,82.889573


Cell 9 - Save Submission

In [9]:
submission.to_csv(
    BASE_DIR / "submission_baseline.csv",
    index=False
)

print("Submission saved.")

Submission saved.
